In [1]:
import pandas as pd
import numpy as np
import os
import re
from sklearn.preprocessing import MinMaxScaler

In [2]:
input_path='data/processed/collected_data_fixed_1.csv'
output_path='data/processed/featured_data_1.csv'

In [3]:
def clean_lyrics(text):
    if not isinstance(text, str) or text.strip() == "":
        return ""
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\d+Embed$', '', text.strip())
    text = text.replace('\n', ' ').strip()
    return text

In [4]:
def engineer_features(input_path,output_path, limit=None):

    df = pd.read_csv(input_path)
    if limit:
        df = df.head(limit)

    print(f"Исходное количество колонок: {len(df.columns)}")

    trash_columns = [
        'spotify_id', 'deezer_id', 'deezer_isrc', 'status_dz', 'itunes_artist_id',
        'itunes_track_view_url', 'itunes_preview_url', 'itunes_artwork',
        'itunes_artist_view_url', 'deezer_preview', 'lastfm_tags'
    ]
    df = df.drop(columns=[c for c in trash_columns if c in df.columns])

    if 'itunes_genre' in df.columns and 'aggregated_genre' in df.columns:
        df['genre_final'] = df['aggregated_genre'].fillna(df['itunes_genre']).fillna('unknown')

    explicit_cols = ['deezer_explicit', 'itunes_explicit']
    df['is_explicit'] = 0
    for col in explicit_cols:
        if col in df.columns:
            df['is_explicit'] = df['is_explicit'] | df[col].fillna(False).astype(int)

    if 'lyrics' in df.columns:
        df['clean_lyrics'] = df['lyrics'].apply(clean_lyrics)
        df['lyrics_word_count'] = df['clean_lyrics'].apply(lambda x: len(x.split()))
    else:
        df['clean_lyrics'] = ""
        df['lyrics_word_count'] = 0

    numeric_map = {
        'bpm': 120,
        'rank': 0,
        'gain': -10,
        'deezer_artist_fans': 0,
        'itunes_duration_ms': 200000
    }

    for col, default in numeric_map.items():
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            fill_value = df[col].median() if df[col].notna().any() else default
            df[col] = df[col].fillna(fill_value)

    scaler = MinMaxScaler()
    cols_to_scale = ['bpm', 'rank', 'gain', 'lyrics_word_count', 'deezer_artist_fans']
    available_scale = [c for c in cols_to_scale if c in df.columns]

    if available_scale:
        scaled_data = scaler.fit_transform(df[available_scale])
        scaled_df = pd.DataFrame(scaled_data, columns=[f'scaled_{c}' for c in available_scale])
        df = pd.concat([df.reset_index(drop=True), scaled_df], axis=1)

    def detect_vibe(row):
        genre = str(row.get('genre_final', '')).lower()
        lyrics = str(row.get('clean_lyrics', '')).lower()
        combined = f"{genre} {lyrics}"

        if any(w in combined for w in ['chill', 'relax', 'slow', 'ambient', 'lofi']): return 'chill'
        if any(w in combined for w in ['hard', 'aggressive', 'metal', 'war', 'fight', 'power']): return 'aggressive'
        if any(w in combined for w in ['dance', 'club', 'pop', 'energy', 'party']): return 'high_energy'
        return 'neutral'

    df['inferred_vibe'] = df.apply(detect_vibe, axis=1)

    threshold = len(df) * 0.9
    df = df.dropna(axis=1, thresh=len(df) - threshold)

    cols_to_remove = ['itunes_genre', 'aggregated_genre', 'deezer_explicit', 'itunes_explicit', 'lyrics']
    df = df.drop(columns=[c for c in cols_to_remove if c in df.columns])

    for col in df.columns:
        if df[col].nunique() <= 1 and col != 'target':
            df = df.drop(columns=[col])

    df.to_csv(output_path, index=False)
    print(f"Осталось колонок: {len(df.columns)}")
    print(f"Колонки: {list(df.columns)}")
    return df

In [5]:
engineer_features(input_path, output_path)

Исходное количество колонок: 28
Осталось колонок: 20
Колонки: ['track_name', 'artist_name', 'target', 'bpm', 'rank', 'gain', 'deezer_artist_fans', 'itunes_release_date', 'itunes_duration_ms', 'itunes_collection_name', 'genre_final', 'is_explicit', 'clean_lyrics', 'lyrics_word_count', 'scaled_bpm', 'scaled_rank', 'scaled_gain', 'scaled_lyrics_word_count', 'scaled_deezer_artist_fans', 'inferred_vibe']


,track_name,artist_name,target,bpm,rank,gain,deezer_artist_fans,itunes_release_date,itunes_duration_ms,itunes_collection_name,genre_final,is_explicit,clean_lyrics,lyrics_word_count,scaled_bpm,scaled_rank,scaled_gain,scaled_lyrics_word_count,scaled_deezer_artist_fans,inferred_vibe
0,Purple Leaves,Dosi,Zen / Lo-Fi Recovery,0.0,116890.0,-12.1,325.0,2025-07-30T12:00:00Z,196000.0,A State of Mind,Hip-Hop/Rap,0,,0,0.000000,0.083027,0.324022,0.000000,0.000016,neutral
1,Deeper,Dosi,Zen / Lo-Fi Recovery,0.0,112642.0,-16.2,144.0,2025-07-30T12:00:00Z,172000.0,A State of Mind,Hip-Hop/Rap,0,,0,0.000000,0.078556,0.094972,0.000000,0.000007,neutral
2,Raindrops,Dosi,Zen / Lo-Fi Recovery,0.0,121615.0,-10.9,325.0,2025-07-30T12:00:00Z,187000.0,A State of Mind,Hip-Hop/Rap,0,,0,0.000000,0.088001,0.391061,0.000000,0.000016,neutral
3,Canal St.,A$AP Rocky ft. Bones,Zen / Lo-Fi Recovery,136.0,458964.0,-11.1,2077041.0,2015-05-26T07:00:00Z,227502.0,AT.LONG.LAST.A$AP,Hip-Hop/Rap,1,"Yeah Live through the strugglin', life's a ev...",662,0.674336,0.443078,0.379888,0.719565,0.113794,aggressive
4,Bohemian Rhapsody,Queen,Golden Era (Old School),141.1,959624.0,-12.1,12690782.0,1975-10-31T12:00:00Z,355145.0,"Greatest Hits I, II & III: The Platinum Collec...",Rock,0,Is this the real life? Is this just fantasy? C...,391,0.699623,0.970049,0.324022,0.425000,0.695291,neutral
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660,Light Echoes,Dosi,Zen / Lo-Fi Recovery,0.0,120666.0,-12.8,325.0,2025-07-30T12:00:00Z,191000.0,A State of Mind,Hip-Hop/Rap,0,,0,0.000000,0.087002,0.284916,0.000000,0.000016,neutral
661,Moonlit,Dosi,Zen / Lo-Fi Recovery,0.0,38008.0,-7.9,25.0,2025-07-30T12:00:00Z,156000.0,A State of Mind,Hip-Hop/Rap,0,,0,0.000000,0.000000,0.558659,0.000000,0.000000,neutral
662,A State of Mind,Dosi,Zen / Lo-Fi Recovery,0.0,114156.0,-11.6,325.0,2025-07-30T12:00:00Z,184500.0,A State of Mind,Hip-Hop/Rap,0,,0,0.000000,0.080150,0.351955,0.000000,0.000016,neutral
663,Goodwill,Dosi,Zen / Lo-Fi Recovery,0.0,120667.0,-12.5,325.0,2025-07-30T12:00:00Z,125500.0,A State of Mind,Hip-Hop/Rap,0,,0,0.000000,0.087003,0.301676,0.000000,0.000016,neutral
